# 상권 데이터 검수 v1 — 이아인 (2026-08-11)

**왜 만들었나.** 지금까지 검수를 일회용 스크립트로 돌리고 결과만 문서에 적었다.
같은 질문("표본이 몇 건부터 믿을 만한가")을 다시 물으면 매번 새로 짜야 했다.

**검증된 로직은 `etl/inspect_data.py` 에 있다** (notebooks/README 규칙).
이 노트북은 그걸 부르고 **해석만** 적는다. 출력 셀은 커밋 시 지워지므로
숫자를 보려면 직접 실행해야 한다.

집객시설·소득소비 CSV 를 받으면 맨 아래 §6 을 그대로 쓴다.

In [ ]:
import sys
from pathlib import Path

import duckdb
import pandas as pd

ROOT = Path.cwd().parents[1] if Path.cwd().name == "ain" else Path.cwd()
sys.path.insert(0, str(ROOT / "etl"))
import inspect_data as ins  # noqa: E402

pd.set_option("display.width", 200, "display.max_columns", 50)
con = duckdb.connect(str(ROOT / "data" / "panel.duckdb"), read_only=True)

## 1. 무엇이 적재돼 있나

분기가 `20261` 하나뿐인 것을 확인한다. 여러 분기가 섞이면 비중 계산이 틀어진다.

In [ ]:
ins.tables(con)

## 2. 결측·중복

전부 0이어야 한다. 서울시 CSV 는 빈칸 대신 0 을 넣는 편이라 결측이 안 잡히는데,
그래서 §4 의 '전부 0인 칸' 검사가 따로 필요하다 (아파트_가구_수가 그랬다).

In [ ]:
ins.nulls_and_dupes(con, {'area': 'area_cd', 'sales': 'area_cd, category_cd, quarter',
                          'foot': 'area_cd, quarter', 'store': 'area_cd, category_cd, quarter'})

## 3. 객단가를 몇 건부터 믿을 수 있나

**`features.MIN_SALES_CNT = 300` 이 이 표에서 나왔다.**

표본이 적을수록 객단가가 업종 중앙값에서 크게 벗어난다. 300건을 넘으면
p90 편차가 2 아래로 떨어진다. 5천 건 이상에서도 남는 0.24 는
**동네마다 객단가가 실제로 다른 것**이라 더 낮출 수 없는 바닥이다.

이 값을 바꾸려면 이 표를 다시 보고 근거를 남길 것.

In [ ]:
ins.ticket_stability(con)

## 4. 극단값 — 이상치인가 실제인가

**결제 건수를 같이 봐야 판단할 수 있다.**

- `한강성심병원 CS300019, 20건, 1.4억` → 표본 20건. 300건 하한에 걸려 폴백된다.
- `합정역 8번 CS300019, 447건, 3,431만원` → **표본이 충분한데도 크다.**
  업종코드를 확인해야 한다. 실제 고액 업종이면 정상이고, 아니면 데이터 문제다.

⚠️ **아직 이상치 판정 기준이 없다.** 표본 하한만 있다. 남은 숙제.

In [ ]:
ins.ticket_extremes(con, n=10)

## 5. 비중 합계

- **시간대**는 합이 정확히 1 — 미상이 없다.
- **성별·연령**은 중앙 0.988, p1 0.372 — 법인카드처럼 누구인지 모르는 결제가 있다.
  그래서 `features` 가 미상을 빼고 재정규화하고, 원래 합계를 `demo_coverage` 로 남긴다.
  0.5 미만이면 `confidence_reasons` 에 '신뢰도 낮음'이 붙는다 (전체의 1.9%).

In [ ]:
ins.share_sums(con)

## 6. 새 CSV 를 받았을 때 — 적재 코드를 짜기 **전에** 돌린다

지난번 검수에서 스키마를 5군데 고쳐야 했다. 성별·연령 교차가 없다,
합계가 100%가 아니다, 아파트_가구_수가 전부 0이다 같은 것들이 그때 나왔다.

받을 예정: **집객시설-상권**(OA-15580), **소득소비-상권**(OA-21278).

확인할 것:
1. 상권 코드가 우리 `area` 1,650곳과 맞는가 (2024년부터 공간 단위가 바뀌었다는 공지가 있다)
2. 분기 코드가 `20261` 을 포함하는가
3. 전부 0인 칸이 있는가
4. 비중 칸이 있다면 합이 1 인가

In [ ]:
# 파일을 data/raw/ 에 넣고 경로만 바꿔 실행
path = ROOT / "data" / "raw" / "서울시 상권분석서비스(집객시설-상권).csv"
if path.exists():
    r = ins.inspect_csv(path)
    print(f"{r['행수']:,}행 × {r['칸수']}칸")
    print("전부 0인 칸:", r["전부 0인 칸"])
    display(r["칸별"])
else:
    print("아직 안 받았습니다:", path.name)

In [ ]:
# 상권 코드가 우리 것과 맞는지 — 안 맞으면 적재해도 매칭이 안 된다
if path.exists():
    df = ins.read_csv(path)
    key = next((c for c in df.columns if "상권_코드" == c), None)
    if key:
        ours = {r[0] for r in con.execute("SELECT area_cd FROM area").fetchall()}
        theirs = set(df[key].astype(str))
        print(f"겹침 {len(ours & theirs):,} / 우리 {len(ours):,} / 저쪽 {len(theirs):,}")